In [ ]:
import os
import torch
from lightning.pytorch import Trainer

from pyannote.audio import Model, Pipeline
from pyannote.audio.tasks import SpeakerDiarization
from pyannote.database import registry
from pyannote.database import FileFinder
from pyannote.metrics.diarization import DiarizationErrorRate

from types import MethodType
from torch.optim import Adam
from pytorch_lightning.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    RichProgressBar,
)

'VoxCeleb.SpeakerDiarization.TaskDiarization' found in /content/database.yml does not define the 'scope' of speaker labels (file, database, or global). Setting it to 'file'.


In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN=userdata.get('HF_TOKEN')

if HF_TOKEN:
    login(HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("Token is not set. Please save the token first.")

Successfully logged in to Hugging Face!


In [ ]:
HF_TOKEN = ""
device = torch.device("cuda")

In [ ]:
preprocessors = {"audio": FileFinder()}
os.environ["PYANNOTE_DATABASE_CONFIG"] = "./database.yml"
protocol = registry.get_protocol("VoxCeleb.SpeakerDiarization.TaskDiarization", preprocessors=preprocessors)

In [ ]:
model = Model.from_pretrained("pyannote/segmentation-3.0", use_auth_token=HF_TOKEN).to(device)

In [ ]:
task = SpeakerDiarization(
    protocol=protocol,
    duration=model.specifications.duration,
    max_num_speakers=4,
    batch_size=32,
    num_workers=0,
    loss="bce",
)
model.task = task

/usr/local/lib/python3.12/dist-packages/pyannote/audio/tasks/segmentation/speaker_diarization.py:150: UserWarning: `max_num_speakers` has been deprecated in favor of `max_speakers_per_chunk`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pyannote/audio/tasks/segmentation/speaker_diarization.py:154: UserWarning: `loss` has been deprecated and has no effect.
  warnings.warn("`loss` has been deprecated and has no effect.")


In [ ]:
def configure_optimizers(self):
    return Adam(self.parameters(), lr=1e-4)

model.configure_optimizers = MethodType(configure_optimizers, model)

monitor, direction = task.val_monitor
checkpoint = ModelCheckpoint(
    monitor=monitor,
    mode=direction,
    save_top_k=1,
    every_n_epochs=1,
    save_last=False,
    save_weights_only=False,
    filename="{epoch}",
    verbose=False,
)
early_stopping = EarlyStopping(
    monitor=monitor,
    mode=direction,
    min_delta=0.0,
    patience=10,
    strict=True,
    verbose=False,
)

callbacks = [RichProgressBar(), checkpoint, early_stopping]

trainer = Trainer(
    devices=1 ,
    accelerator="cpu",
    max_epochs=20,
    default_root_dir="finetuned_model"
)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(model)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/usr/local/lib/python3.12/dist-packages/pyannote/database/protocol/protocol.py:140: UserWarning: Existing precomputed key "annotation" has been modified by a preprocessor.
  warnings.warn(msg.format(key=key))
/usr/local/lib/python3.12/dist-packages/pyannote/audio/core/model.py:220: UserWarning: Model has been trained for a different task. For fine tuning or transfer learning, it is recommended to train task-dependent layers for a few epochs before training the whole model: ['activation', 'classifier'].
  warnings.warn(msg)


┏━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   ┃ Name              ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃       In sizes ┃                  Out sizes ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0 │ sincnet           │ SincNet          │ 42.6 K │ train │ 960 M │ [1, 1, 160000] │               [1, 60, 589] │
│ 1 │ lstm              │ LSTM             │  1.4 M │ train │     0 │   [1, 589, 60] │    [[1, 589, 256], [[8, 1, │
│   │                   │                  │        │       │       │                │        128], [8, 1, 128]]] │
│ 2 │ linear            │ ModuleList       │ 49.4 K │ train │     0 │              ? │                          ? │
│ 3 │ classifier        │ Linear           │  1.4 K │ train │ 1.7 M │  [1, 589, 128] │               [1, 589, 11] │
│ 4 │ activation        │ LogSoftmax       │      0 │ train │     0 │   [1, 589, 11] │               [1, 589, 11] │
│ 5 │ powerset          │ Powerset         │      0 │ train │     0 │              ? │                          ? │
│ 6 │ validation_metric │ MetricCollection │      0 │ train │     0 │              ? │                          ? │
└───┴───────────────────┴──────────────────┴────────┴───────┴───────┴────────────────┴────────────────────────────┘

Trainable params: 1.5 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.5 M                                                                                                
Total estimated model params size (MB): 5.895                                                                      
Modules in train mode: 30                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 1.0 B

Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

...

/usr/local/lib/python3.12/dist-packages/torch_audiomentations/core/transforms_interface.py:221: UserWarning: 
target_rate is required by Identity. It has been automatically inferred from targets shape to 59. If this is 
incorrect, you can pass it directly.
  warnings.warn(

INFO:pytorch_lightning.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 217, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 469, in advance
    self.epoch_loop.run(self._data_fetcher)
  File

TypeError: object of type 'NoneType' has no len()

In [ ]:
checkpoint_path = "finetuned_model/lightning_logs/version_4/checkpoints/epoch=19-step=4560.ckpt"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

model.load_state_dict(checkpoint["state_dict"])

<All keys matched successfully>

In [1]:
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", token=HF_TOKEN)
pipeline.to(device)

metric = DiarizationErrorRate()

for file in protocol.test():

    # Run the pipeline to get the hypothesis (predicted diarization)
    output = pipeline(file)

    # UEM (Un-Evaluated Map) defines the regions of the audio that are meant to be evaluated
    hypothesis = output.speaker_diarization
    reference = file["annotation"]
    uem = file.get("annotated")

    metric(reference, hypothesis, uem=uem)


# 7. Compute and print the final results
final_der = abs(metric)
print("\n" + "="*40)
print(f"Total Diarization Error Rate (DER): {final_der * 100:.2f}%")
print("="*40)



Total Diarization Error Rate (DER): 9.79%
